# Churn / Segmentation Model — score the new 100,000 customers\n\nTrains a regressor on the **seed 100,000 customers** (frozen, IDs 1–100,000) to predict their churn `Score`, then applies it to the **new 100,000 customers** (IDs 100,001–200,000, from `data/new_customers_features.csv`) and writes the results directly into `FactCustomerSegmentScore`.\n\n**Target:** `Score` (continuous). `PredictedChurn`, `RiskBand` are then derived from the predicted score using the exact same thresholds the original warehouse generator used (`PredictedChurn` if Score > 0.70; `RiskBand` High/Medium/Low at 0.80/0.50), so the new rows are structurally identical to the seed rows. `SegmentName` is a direct pass-through of the customer's own `Segment` (that's what the original generator does too - it isn't modeled). `ModelName`/`Explanation` are fixed metadata, same as the seed data.\n\n**Same caveat as every notebook in this project:** the seed `Score` values are themselves a formula of `CustomerId`, not real churn behavior, so this model mostly learns to reconstruct that formula's relationship to the other synthetic fields. The pipeline (train on seed → predict on new → write to warehouse) is the real deliverable.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from db_utils import bulk_insert, get_connection

FEATURE_COLS = [
    "Age", "Gender", "Region", "CustomerType", "Segment", "CustomerStatus",
    "Balance", "AccountType",
]
TODAY = date.today()

## 1. Load the seed training data (customers 1–100,000)

In [2]:
conn = get_connection()
seed = pd.read_sql(
    """
    SELECT c.CustomerId, c.Age, c.Gender, c.Region, c.CustomerType, c.Segment, c.CustomerStatus,
           a.Balance, a.AccountType,
           s.Score
    FROM dbo.DimCustomer c
    JOIN dbo.FactCustomerAccount a ON a.CustomerId = c.CustomerId
    JOIN dbo.FactCustomerSegmentScore s ON s.CustomerId = c.CustomerId
    WHERE c.CustomerId <= 100000;
    """,
    conn,
)
conn.close()
print(seed.shape)
seed.head()

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_37787/1873988791.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seed = pd.read_sql(


(100000, 10)


,CustomerId,Age,Gender,Region,CustomerType,Segment,CustomerStatus,Balance,AccountType,Score
0,1,30,Male,Abuja,Retail,Premium Retail,Active,50125.5,Savings Account,0.121
1,18,47,Female,Port Harcourt,Corporate,SME Corporate,Active,52259.0,Current Account,0.138
2,44,28,Female,Enugu,SME,Growth SME,Active,55522.0,SME Current Account,0.164
3,67,51,Male,Kano,Retail,Youth Retail,Active,58408.5,Savings Account,0.187
4,68,52,Female,Enugu,SME,Growth SME,Active,58534.0,SME Current Account,0.188


## 2. Load the new customers to predict on, and encode both sets consistently\n\nTrain and predict data are loaded separately, so categorical columns are one-hot encoded together and then split back apart — otherwise the two sets could end up with mismatched dummy columns.

In [3]:
new_customers = pd.read_csv("data/new_customers_features.csv")
print(new_customers.shape)

combined = pd.concat(
    [seed[FEATURE_COLS], new_customers[FEATURE_COLS]],
    keys=["seed", "new"],
)
combined_encoded = pd.get_dummies(combined, drop_first=True)

X_seed = combined_encoded.loc["seed"].reset_index(drop=True)
X_new = combined_encoded.loc["new"].reset_index(drop=True)
y_seed = seed["Score"].reset_index(drop=True)

print("Encoded feature count:", X_seed.shape[1])

(100000, 15)
Encoded feature count: 23


## 3. Train and evaluate (held-out split on the seed data)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X_seed, y_seed, test_size=0.2, random_state=42)

eval_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))
print("R2:", round(r2_score(y_test, y_pred), 4))

MAE: 0.1992
R2: 0.0085


## 4. Refit on all seed data, predict onto the new 100,000

In [5]:
final_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
final_model.fit(X_seed, y_seed)

predicted_score = np.clip(final_model.predict(X_new), 0, 1).round(4)

results = pd.DataFrame({
    "CustomerId": new_customers["CustomerId"],
    "ModelName": "Churn_Model",
    "Score": predicted_score,
    "ModelDate": TODAY,
    "PredictedChurn": (predicted_score > 0.70).astype(int),
    "SegmentName": new_customers["Segment"],
    "RiskBand": np.select(
        [predicted_score > 0.80, predicted_score > 0.50],
        ["High", "Medium"],
        default="Low",
    ),
    "Explanation": "Customer activity and product utilization trend",
})

print(results["RiskBand"].value_counts())
results.head()

RiskBand
Medium    67918
Low       32082
Name: count, dtype: int64


,CustomerId,ModelName,Score,ModelDate,PredictedChurn,SegmentName,RiskBand,Explanation
0,100001,Churn_Model,0.5598,2026-07-23,0,Service SME,Medium,Customer activity and product utilization trend
1,100002,Churn_Model,0.5476,2026-07-23,0,Youth Retail,Medium,Customer activity and product utilization trend
2,100003,Churn_Model,0.5580,2026-07-23,0,Manufacturing SME,Medium,Customer activity and product utilization trend
3,100004,Churn_Model,0.5195,2026-07-23,0,Mass Retail,Medium,Customer activity and product utilization trend
4,100005,Churn_Model,0.5195,2026-07-23,0,Premium Retail,Medium,Customer activity and product utilization trend


## 5. Write into the live warehouse

In [6]:
cols = ["CustomerId", "ModelName", "Score", "ModelDate", "PredictedChurn", "SegmentName", "RiskBand", "Explanation"]

conn = get_connection()
n = bulk_insert(conn, "dbo.FactCustomerSegmentScore", cols, list(results[cols].itertuples(index=False, name=None)))
conn.close()
print(f"Inserted {n:,} rows into FactCustomerSegmentScore")

Inserted 100,000 rows into FactCustomerSegmentScore


In [7]:
conn = get_connection()
check = pd.read_sql(
    "SELECT COUNT(*) AS NewScoreRows FROM dbo.FactCustomerSegmentScore WHERE CustomerId > 100000;",
    conn,
)
conn.close()
check

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_37787/3119242535.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql(


,NewScoreRows
0,100000
